In [1]:
import pandas as pd
import numpy as np
import importlib
import gc
import sys
import warnings
warnings.filterwarnings("ignore")

In [ ]:
sys.path.append('../../notebooks/entregable/scripts')
import dataset
import preprocesamiento
import target
import feature_engineering
importlib.reload(dataset)
importlib.reload(preprocesamiento)
importlib.reload(target)
importlib.reload(feature_engineering)

In [ ]:
df = pd.read_csv("../../data/preprocessed/base.csv", sep=',')
df.shape

(2945818, 13)

In [ ]:
#### COMBINATORIA ####
data = dataset.combinatoria_periodo_producto()
data['periodo'] = data['periodo'].dt.year * 100 + data['periodo'].dt.month
data.shape

(44388, 2)

In [ ]:
data

,product_id,periodo
0,20524,201701
1,20524,201702
2,20524,201703
3,20524,201704
4,20524,201705
...,...,...
44383,20770,201908
44384,20770,201909
44385,20770,201910
44386,20770,201911


Filtramos los 780 productos

In [ ]:
productos_ok = pd.read_csv("../../data/raw/product_id_apredecir201912.csv", sep="\t")
data = data[data['product_id'].isin(productos_ok['product_id'].unique())]
data

,product_id,periodo,cat1,cat2,cat3,brand,sku_size,stock_final,tn,plan_precios_cuidados,cust_request_qty,cust_request_tn
0,20524,201701,HC,VAJILLA,Cristalino,Importado,500,NaN,6.48085,0.0,148.0,6.48085
1,20524,201702,HC,VAJILLA,Cristalino,Importado,500,NaN,3.99755,0.0,121.0,3.99755
2,20524,201703,HC,VAJILLA,Cristalino,Importado,500,NaN,7.14711,0.0,119.0,7.14711
3,20524,201704,HC,VAJILLA,Cristalino,Importado,500,NaN,6.82163,0.0,124.0,6.82163
4,20524,201705,HC,VAJILLA,Cristalino,Importado,500,NaN,9.25949,0.0,161.0,9.25949
...,...,...,...,...,...,...,...,...,...,...,...,...
28075,20127,201908,HC,ROPA LAVADO,Liquido,ROPEX2,3000,NaN,0.00000,NaN,NaN,NaN
28076,20127,201909,HC,ROPA LAVADO,Liquido,ROPEX2,3000,-0.03361,12.80399,0.0,14.0,12.80399
28077,20127,201910,HC,ROPA LAVADO,Liquido,ROPEX2,3000,55.69684,186.81735,0.0,128.0,187.47827
28078,20127,201911,HC,ROPA LAVADO,Liquido,ROPEX2,3000,30.54813,463.80054,0.0,333.0,469.63684


Mergeamos

In [ ]:
#### MERGE CON PRODUCTOS ####
productos = pd.read_csv("../../data/raw/tb_productos.csv", sep='\t')
productos = productos.drop_duplicates(subset=['product_id'], keep='first')
data = data.merge(productos, how='left', on="product_id")
del productos

#### MERGE CON STOCKS ####
stocks = pd.read_csv("../../data/raw/tb_stocks.csv", sep='\t')
stocks = stocks.groupby(by=["periodo", "product_id"]).agg({"stock_final": "sum"}).reset_index()
data = data.merge(stocks, how='left', on=['periodo', 'product_id'])
del stocks

#### MERGE CON SELLIN ####
sellin = pd.read_csv("../../data/raw/sell-in.csv", sep='\t')
sellin = sellin.groupby(by=["periodo","product_id"]).agg({"tn":"sum", "plan_precios_cuidados":"sum", "cust_request_qty":"sum", "cust_request_tn":"sum"}).reset_index()
data = data.merge(sellin, how='left', on=['periodo', 'product_id'])
del sellin
gc.collect()

67

Completamos con ceros

In [ ]:
#### COMPLETO TN CON CEROS ####
####  ¿cuantos?
print(f"Total de periodos con Nan debido a la combinatoria periodo_x_producto: {data['tn'].isna().sum()}")
#### Lo completo con ceros
data['tn'] = data['tn'].fillna(0)

Total de periodos con Nan debido a la combinatoria periodo_x_producto: 5731


Guardamos

In [ ]:
#### GUARDAR DATAFRAME ####
data.to_csv("./datasets/periodo_x_producto.csv", index=False, sep=',', encoding='utf-8')

AutoARIMA

```python
import statsforecast.models as m
print(dir(m))

In [ ]:
data = pd.read_csv("./periodo_x_producto.csv", sep=',')

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, AutoTheta, Naive, SeasonalNaive, HoltWinters
from joblib import Parallel, delayed  # Para paralelizar (opcional)

ModuleNotFoundError: No module named 'statsforecast'

In [ ]:
# Creo DF
ts = data[['product_id','periodo', 'tn']].copy()

# Convertir 'periodo' (yyyymm) a datetime
ts['periodo_dt'] = pd.to_datetime(ts['periodo'].astype(str), format='%Y%m')
ts = ts.sort_values(['product_id', 'periodo_dt'])  # Ordenar por producto y fecha

ts.drop(columns=['periodo'], inplace=True)  # Eliminar columna temporal
ts.rename(columns={'tn': 'y', 'periodo_dt':'ds', 'product_id':'unique_id'}, inplace=True)  # Renombrar columna de tn

In [ ]:
models = [
    AutoARIMA(season_length=12),
    AutoETS(season_length=12),
    AutoTheta(season_length=12)
]


predictions = {}

for product_id in productos_ok['product_id'].unique():
    df_product = ts[ts['unique_id'] == product_id].copy()

    # Añadir columna unique_id (requerida por StatsForecast)
    df_product['unique_id'] = product_id

    # Seleccionar columnas necesarias y ordenar por fecha
    df_product = df_product[['unique_id', 'ds', 'y']].sort_values('ds')

    if len(df_product) >= 12:
        try:
            sf = StatsForecast(models=models, freq='MS', n_jobs=-1)
            pred = sf.forecast(h=2, df=df_product)

            # Obtener la última predicción (h=2)
            pred_feb2020 = pred.sort_values('ds').iloc[[-1]]

            # Calcular promedio de modelos
            mean_pred = pred_feb2020[['AutoARIMA', 'AutoETS', 'AutoTheta']].mean(axis=1).iloc[0]
            predictions[product_id] = mean_pred

            print(f"Producto {product_id}: Modelo ajustado")

        except Exception as e:
            print(f"Error en producto {product_id}: {str(e)}")
            predictions[product_id] = None
    else:
        print(f"Producto {product_id}: Insuficientes datos ({len(df_product)} observaciones)")
        predictions[product_id] = None

# Convertir a DataFrame
df_predictions = pd.DataFrame({
    'product_id': predictions.keys(),
    'prediccion_mes+2': predictions.values()
})

Producto 20001: Modelo ajustado
Producto 20002: Modelo ajustado
Producto 20003: Modelo ajustado
Producto 20004: Modelo ajustado
Producto 20005: Modelo ajustado
Producto 20006: Modelo ajustado
Producto 20007: Modelo ajustado
Producto 20008: Modelo ajustado
Producto 20009: Modelo ajustado
Producto 20010: Modelo ajustado
Producto 20011: Modelo ajustado
Producto 20012: Modelo ajustado
Producto 20013: Modelo ajustado
Producto 20014: Modelo ajustado
Producto 20015: Modelo ajustado
Producto 20016: Modelo ajustado
Producto 20017: Modelo ajustado
Producto 20018: Modelo ajustado
Producto 20019: Modelo ajustado
Producto 20020: Modelo ajustado
Producto 20021: Modelo ajustado
Producto 20022: Modelo ajustado
Producto 20023: Modelo ajustado
Producto 20024: Modelo ajustado
Producto 20025: Modelo ajustado
Producto 20026: Modelo ajustado
Producto 20027: Modelo ajustado
Producto 20028: Modelo ajustado
Producto 20029: Modelo ajustado
Producto 20030: Modelo ajustado
Producto 20031: Modelo ajustado
Producto

In [ ]:
df_predictions.to_csv("./outputs/autoarima.csv", index=False, sep=',', encoding='utf-8')

Prophet

In [ ]:
import pandas as pd
from prophet import Prophet
from tqdm import tqdm

# ---------------------
# 🛠 Preparación inicial
# ---------------------

# Creo DF
df = data[['product_id','periodo', 'tn']].copy()

# Convertir 'periodo' (yyyymm) a datetime
df['periodo_dt'] = pd.to_datetime(df['periodo'].astype(str), format='%Y%m')
df = df.sort_values(['product_id', 'periodo_dt'])  # Ordenar por producto y fecha

df.drop(columns=['periodo'], inplace=True)  # Eliminar columna temporal
df.rename(columns={'tn': 'y', 'periodo_dt':'ds'}, inplace=True)  # Renombrar columna de tn

productos_ok = pd.read_csv("./product_id_apredecir201912.csv", sep="\t")



# Para guardar las predicciones
resultados = []

# ---------------------
#  Loop por producto
# ---------------------
for product_id in productos_ok['product_id'].unique():
    df_prod = df[df['product_id'] == product_id].sort_values('ds')

    if len(df_prod) < 6:
        continue  # opcional: saltear series demasiado cortas

    # Entrenar Prophet
    model = Prophet(yearly_seasonality=True)
    model.fit(df_prod[['ds', 'y']])

    # Predecir mes +2
    ultima_fecha = df_prod['ds'].max()
    future = model.make_future_dataframe(periods=2, freq='MS')
    future = future[future['ds'] > ultima_fecha]  # solo fechas futuras
    pred = model.predict(future)

    # Obtener solo la predicción de mes+2
    pred_mes2 = pred.tail(1)

    resultados.append({
        'product_id': product_id,
        'fecha_predicha': pred_mes2['ds'].values[0],
        'yhat': pred_mes2['yhat'].values[0],
        'yhat_lower': pred_mes2['yhat_lower'].values[0],
        'yhat_upper': pred_mes2['yhat_upper'].values[0]
    })

# ---------------------
# 📊 Resultados finales
# ---------------------
df_predicciones = pd.DataFrame(resultados)
print(df_predicciones.head())


Streaming output truncated to the last 5000 lines.
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['/usr/local/lib/python3.11/dist-packages/prophet/stan_model/prophet_model.bin', 'random', 'seed=99176', 'data', 'file=/tmp/tmp2f6ae0sm/6kf2skra.json', 'init=/tmp/tmp2f6ae0sm/aa8m1oyd.json', 'output', 'file=/tmp/tmp2f6ae0sm/prophet_modelxi2ve4lr/prophet_model-20250714230441.csv', 'method=optimize', 'algorithm=newton', 'iter=10000']
23:04:41 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing
23:04:41 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing
INFO:prophet:Disabling weekly seasonality. Run prophet with weekly_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
DEBUG:cmdstanpy:input tempfile: /tmp/tmp2f6ae0sm/te6p6cck.json
DEBUG:cmdstanpy:input tempfile: /tmp/tmp2f6ae0sm/4k71kygo.json
DEBUG:cmds

   product_id fecha_predicha         yhat   yhat_lower   yhat_upper
0       20001     2020-02-01  1362.070852  1142.126578  1581.398773
1       20002     2020-02-01  1117.738591   953.416011  1310.537726
2       20003     2020-02-01   624.334799   486.470776   766.102854
3       20004     2020-02-01   230.298529   105.875275   342.459039
4       20005     2020-02-01   279.618412   174.914519   375.508762


Neural Prophet

In [3]:
from neuralprophet import NeuralProphet
from tqdm import tqdm

# ---------------------
# 🛠 Preparación inicial
# ---------------------
# Levanto
data = pd.read_csv("./periodo_x_producto.csv", sep=',')

# Creo DF
df = data[['product_id','periodo', 'tn']].copy()

# Convertir 'periodo' (yyyymm) a datetime
df['periodo_dt'] = pd.to_datetime(df['periodo'].astype(str), format='%Y%m')
df = df.sort_values(['product_id', 'periodo_dt'])  # Ordenar por producto y fecha

df.drop(columns=['periodo'], inplace=True)  # Eliminar columna temporal
df.rename(columns={'tn': 'y', 'periodo_dt':'ds'}, inplace=True)  # Renombrar columna de tn

productos_ok = pd.read_csv("./product_id_apredecir201912.csv", sep="\t")

# Lista para guardar predicciones
predicciones = []


# ---------------------
# 🔁 Loop por producto
# ---------------------
for product_id in productos_ok['product_id'].unique():
    df_prod = df[df['product_id'] == product_id].sort_values('ds')

    if len(df_prod) < 6:
        continue  # Salta productos con muy pocos datos

    try:
        # Definir modelo NeuralProphet
        model = NeuralProphet(
            yearly_seasonality=True,
            weekly_seasonality=False,
            daily_seasonality=False,
            seasonality_mode='additive'  # importante si hay ceros
        )

        # Entrenar modelo
        model.fit(df_prod[['ds', 'y']], freq='MS', progress='off')

        # Crear fechas futuras para mes+2
        future = model.make_future_dataframe(df_prod[['ds', 'y']], periods=2)
        forecast = model.predict(future)

        # Extraer predicción del mes+2 (última fila)
        forecast_mes2 = forecast.tail(1)

        predicciones.append({
            'product_id': product_id,
            'fecha_predicha': forecast_mes2['ds'].values[0],
            'yhat1': forecast_mes2['yhat1'].values[0]
        })
    except Exception as e:
        print(f"⚠️ Producto {product_id} falló: {e}")

# ---------------------
# 📊 Resultados finales
# ---------------------
df_predicciones = pd.DataFrame(predicciones)
print(df_predicciones.head())

ERROR:NP.plotly:Importing plotly failed. Interactive plots will not work.
ERROR:NP.plotly:Importing plotly failed. Interactive plots will not work.
WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency MS corresponds to [97.222]% of the data.
INFO:NP.df_utils:Major frequency MS corresponds to [97.222]% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - MS
INFO:NP.df_utils:Defined frequency is equal to major frequency - MS
INFO - (NP.config.init_data_params) - Setting normalization to global as only one dataframe provided for training.
INFO:NP.config:Setting normalization to global as only one dataframe provided for training.
INFO - (NP.config.set_auto_batch_epoch) - Auto-set batch_size to 8
INFO:NP.config:Auto-set batch_size to 8
INFO - (NP.config.set_auto_batch_epoch) - Auto-set epochs to 380
INFO:NP.config:Auto-s

Training: |          | 0/? [00:00<?, ?it/s]

WARNING - (NP.config.set_lr_finder_args) - Learning rate finder: The number of batches (5) is too small than the required number                     for the learning rate finder (203). The results might not be optimal.


Finding best initial lr:   0%|          | 0/203 [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

...

⚠️ Producto 20003 falló: name 'exit' is not defined


Training: |          | 0/? [00:00<?, ?it/s]

WARNING - (NP.config.set_lr_finder_args) - Learning rate finder: The number of batches (5) is too small than the required number                     for the learning rate finder (203). The results might not be optimal.


Finding best initial lr:   0%|          | 0/203 [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

In [3]:
!pip install neuralprophet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.8/145.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 119.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.4/825.4 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 131.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━